# 03 - Hotspot and Anomaly Detection Demo

This notebook runs the full control-plane inference stack, inspects hotspot scores, and visualizes anomaly evidence.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
from IPython.display import display

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.control_plane.inference_hub import InferenceHub
from src.models.anomaly.statistical_detector import VolumeMetrics

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "io_features.parquet"
df = pd.read_parquet(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
latest_ts = df["timestamp"].max()
print("Latest timestamp:", latest_ts)

hub = InferenceHub(project_root=PROJECT_ROOT)
print("InferenceHub initialized.")

In [ ]:
# Run analyze_volume for all 50 volumes at the latest timestamp
results = []
for volume_id in sorted(df["volume_id"].drop_duplicates().tolist()):
    analysis = hub.analyze_volume(volume_id, latest_ts)
    node_id = hub.topology.get_node_of_volume(volume_id)
    results.append({
        "volume_id": volume_id,
        "hotspot_score": analysis["hotspot_score"],
        "workload_type": analysis["workload_type"],
        "node_id": node_id,
        "analysis": analysis,
    })

hotspot_df = pd.DataFrame(results).sort_values("hotspot_score", ascending=False).reset_index(drop=True)
display(hotspot_df[["volume_id", "hotspot_score", "workload_type", "node_id"]])

def severity_from_score(score: float) -> str:
    if score < 40:
        return "green"
    if score < 60:
        return "yellow"
    if score < 80:
        return "orange"
    return "red"

hotspot_df["severity"] = hotspot_df["hotspot_score"].apply(severity_from_score)

In [ ]:
# Hotspot ranking bar chart
color_map = {"green": "#2ca02c", "yellow": "#ffdd57", "orange": "#ff7f0e", "red": "#d62728"}
bar_colors = hotspot_df["severity"].map(color_map)
plt.figure(figsize=(14, 10))
plt.barh(hotspot_df["volume_id"], hotspot_df["hotspot_score"], color=bar_colors)
plt.gca().invert_yaxis()
plt.title("All 50 Volumes Sorted by Hotspot Score")
plt.xlabel("hotspot_score")
plt.ylabel("volume_id")
plt.tight_layout()
plt.show()

print("Top 10 hotspot volumes:")
display(hotspot_df[["volume_id", "hotspot_score", "workload_type", "node_id", "severity"]].head(10))

In [ ]:
# LSTM input sequences for the top 3 hotspot volumes
top3 = hotspot_df.head(3).copy()
fig, axes = plt.subplots(nrows=len(top3), ncols=1, figsize=(14, 10), sharex=True)
if len(top3) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, top3.iterrows()):
    volume_id = row["volume_id"]
    seq = hub.get_lstm_sequence(volume_id, latest_ts)
    steps = np.arange(1, seq.shape[0] + 1)
    ax.plot(steps, seq[:, 0], marker="o", label="total_iops")
    ax2 = ax.twinx()
    ax2.plot(steps, seq[:, 1], marker="s", color="tab:red", label="avg_latency_us")
    ax.set_title(f"{volume_id} | hotspot_score={row['hotspot_score']:.2f}")
    ax.set_ylabel("total_iops")
    ax2.set_ylabel("avg_latency_us")
    ax.grid(True, alpha=0.3)

plt.xlabel("LSTM step (oldest → newest)")
plt.tight_layout()
plt.show()

# Noisy-neighbor relationships
nn_rows = []
for _, row in hotspot_df.iterrows():
    victims = row["analysis"].get("noisy_neighbor_victims", {})
    for victim_id, impact_score in victims.items():
        nn_rows.append({
            "aggressor": row["volume_id"],
            "victim": victim_id,
            "impact_score": impact_score,
        })

if nn_rows:
    nn_df = pd.DataFrame(nn_rows).sort_values(["aggressor", "impact_score"], ascending=[True, False])
    display(nn_df)
else:
    print("No noisy-neighbor aggressor/victim relationships were detected at the latest timestamp.")

In [ ]:
# Per-volume score breakdown for the top 3 hotspot volumes
breakdown_rows = []
for volume_id in top3["volume_id"].tolist():
    row = hub.get_raw_feature_row(volume_id, latest_ts)
    metrics = VolumeMetrics(
        timestamp=latest_ts,
        total_iops=float(row["total_iops"]),
        avg_latency_us=float(row["avg_latency_us"]),
        total_throughput_mbps=float(row["total_throughput_mbps"]),
        read_latency_p99_us=float(row["read_latency_p99_us"]),
        write_latency_p99_us=float(row["write_latency_p99_us"]),
        capacity_used_pct=float(row.get("capacity_used_pct", 0.0)),
    )
    raw_features = row[hub.ensemble.FEATURE_COLS].values.astype(np.float32)
    sequence = hub.get_lstm_sequence(volume_id, latest_ts)
    stat_score, _ = hub.ensemble.stat_detector.detect_hotspot(volume_id, metrics)
    if_score, _ = hub.ensemble._run_if(raw_features)
    lstm_score, _ = hub.ensemble._run_lstm_single(sequence)
    ensemble_score = hub.ensemble._fuse_scores(stat_score, if_score, lstm_score)
    breakdown_rows.append({
        "volume_id": volume_id,
        "stat_score": stat_score,
        "if_score": if_score,
        "lstm_score": lstm_score,
        "ensemble_score": ensemble_score,
    })

breakdown_df = pd.DataFrame(breakdown_rows)
display(breakdown_df)

plot_df = breakdown_df.melt(id_vars="volume_id", value_vars=["stat_score", "if_score", "lstm_score", "ensemble_score"], var_name="score_type", value_name="score")
plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x="volume_id", y="score", hue="score_type")
plt.title("Score Breakdown for Top 3 Hotspot Volumes")
plt.xlabel("volume_id")
plt.ylabel("score")
plt.tight_layout()
plt.show()